# Finding an image from a long multi-line description with CLIP

Each query is a long description with one line per region of the image, and the goal is to find the matching image in a gallery. I embed every line separately with CLIP ViT-B/16, drop near-duplicate lines and rank the images by the mean of the top-k line similarities. MRR@10 on the train set was 0.515.

#||IMPORTS||

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
import pandas as pd

Mounted at /content/drive


In [2]:
import shutil
import os
import zipfile

listy = ['p2_tst_2025_starter.zip', 'p2_tst_2025_train_data.zip', 'p2_tst_2025_test_data.zip']
listy_want = ['starter.zip', 'train.zip', 'test.zip']

for i in range(3):

  source_path = '/content/drive/MyDrive/' + listy[i]
  destination_path = '/content/' + listy_want[i]

  shutil.copy(source_path, destination_path)

  zip_path = "/content/" + listy_want[i]
  extract_path = "/content/" + listy_want[i][0:len(listy_want[i])-4]

  os.makedirs(extract_path, exist_ok=True)

  with zipfile.ZipFile(zip_path, 'r') as l:
      l.extractall(extract_path)

#0. Setup
We use HuggingFace transformers to load CLIP.

In [3]:
# !pip install transformers torch torchvision pillow pandas numpy
import os, numpy as np, pandas as pd, torch
from pathlib import Path
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DATA = Path('/content/')
TRAIN_DIR = DATA/'train'
TEST_DIR  = DATA/'test'

def get_device():
    if torch.cuda.is_available(): return 'cuda'
    if torch.backends.mps.is_available(): return 'mps'
    return 'cpu'
device = get_device(); print('device:', device)

device: cuda


In [4]:
MODEL_DIR = '/content/starter/models/clip-vit-base-patch16'
src = MODEL_DIR if os.path.isdir(MODEL_DIR) else 'openai/clip-vit-base-patch16'  # hub id needs internet
print('loading CLIP from:', src)
model = CLIPModel.from_pretrained(src).to(device).eval()
processor = CLIPProcessor.from_pretrained(src)

loading CLIP from: /content/starter/models/clip-vit-base-patch16


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

#||LOAD DATA||

In [5]:
train = pd.read_csv(TRAIN_DIR/'train.csv')
test  = pd.read_csv(TEST_DIR/'test.csv')
print('train pairs:', len(train), '| test queries:', len(test))

gallery_paths = sorted((TEST_DIR/'images').glob('*.jpg'))
gallery_ids   = [p.stem for p in gallery_paths]
train_paths = sorted((TRAIN_DIR/'images').glob('*.jpg'))
train_ids   = [p.stem for p in train_paths]

print('test gallery images:', len(gallery_ids), '| train images:', len(train_ids))

train pairs: 500 | test queries: 100
gallery images: 500

query 29 | word count: 1186
Jeep section: Part of a white-colored Jeep.
Poles and car: A couple of silver poles and a white car.
Tree branches: A cluster of tree branches that have green leaves hanging off of them.
Barrier section: The triangular top of a yellow barrier. It has thin beams going down and across it, with a chain link fence going across it.
Trees: Several large trees that stand in the background. All of the trees have green leaves.
Sign: A rectangular white sign that is folded along its left side. There are images of people, and illegible lettering, printed on the sign.
Motorcycle back: The back side of a b ...


# 6: ||ENCODING FUNCTIONS||

In [6]:
@torch.no_grad()
def encode_images(paths, bs=32):
    feats=[]
    for i in range(0, len(paths), bs):
        imgs = [Image.open(p).convert('RGB') for p in paths[i:i+bs]]
        inp = processor(images=imgs, return_tensors='pt').to(device)
        pooled = model.vision_model(pixel_values=inp['pixel_values']).pooler_output
        e = model.visual_projection(pooled).float()
        e = e/e.norm(dim=-1,keepdim=True).clamp_min(1e-12)
        feats.append(e.cpu())
    return torch.cat(feats)

img_emb_test = encode_images(gallery_paths); print('gallery embeddings:', tuple(img_emb_test.shape))
img_emb_train = encode_images(train_paths); print('gallery embeddings:', tuple(img_emb_train.shape))

gallery embeddings: (600, 512)
gallery embeddings: (500, 512)


In [116]:
@torch.no_grad()

def parse_lines(desc):
    lines = [l.strip() for l in str(desc).split('\n')]
    return [l for l in lines if l]

@torch.no_grad()
def encode_query(text):

    lines1 = [ line.strip() for line in str(text).split('\n') if line.strip()]

    inp = processor(text=lines1, return_tensors='pt',
                    padding=True, truncation=True, max_length=77).to(device)

    pooled = model.text_model(**inp).pooler_output
    e = model.text_projection(pooled).float()

    return (e/e.norm(dim=-1,keepdim=True).clamp_min(1e-12)).cpu()

In [117]:
train_line_lists = [d for d in train['description']]

train_query_embs = []
idx = 0
for line in train_line_lists:
    train_query_embs.append(encode_query(line))
print('encoded line-sets for', len(train_query_embs), 'train dossiers')

encoded line-sets for 500 train dossiers


In [118]:
test_line_lists = [d for d in test['description']]

test_query_embs = []
idx = 0
for line in test_line_lists:
    test_query_embs.append(encode_query(line))
print('encoded line-sets for', len(test_query_embs), 'train dossiers')

encoded line-sets for 100 train dossiers


#|| DEDUPLICATED LINES||

In [119]:
def deduplicate(line_emb, threshold=0.93):
  kept = []
  mask = np.zeros(line_emb.shape[0], dtype=bool)
  sims = (line_emb @ line_emb.T).numpy()
  for i in range(line_emb.shape[0]):
      if not kept or max(sims[i, j] for j in kept) < threshold:
          kept.append(i)
  return line_emb[kept]

In [120]:
train_line_embs = []
for line in train_query_embs:
    train_line_embs.append(deduplicate(line))

test_line_embs = []
for line in test_query_embs:
    test_line_embs.append(deduplicate(line))

In [125]:
def score_topk_mean(line_emb, gallery_emb, k):
    sims = (line_emb @ gallery_emb.T).numpy()  # (N, M)
    k = min(k, sims.shape[0])
    topk = np.sort(sims, axis=0)[-k:, :]
    return topk.mean(axis=0)

In [126]:
def rank_top10(line_emb, gallery_emb, gallery_id_list, k):
    scores = score_topk_mean(line_emb, gallery_emb, k=k)
    order = np.argsort(-scores)[:10]
    return [gallery_id_list[j] for j in order]

In [127]:
#%%
def mrr_at_10(rank_of_correct):
    return 1.0/rank_of_correct if rank_of_correct is not None and rank_of_correct <= 10 else 0.0

def evaluate_config(train_query_embs, k, zscore, n_eval=None):
    n_eval = len(train_query_embs)
    rrs = []
    for i in range(n_eval):
        line_emb = train_query_embs[i]
        scores = score_topk_mean(line_emb, img_emb_train, k=10)
        order = np.argsort(-scores)

        true_id = train.iloc[i]['image_id']

        rank = int(np.where(np.array(train_ids)[order] == true_id)[0][0]) + 1
        rrs.append(mrr_at_10(rank))
    return float(np.mean(rrs))

results = {}
for zscore in [False, True]:
    for k in [5, 10, 15, 20, 30, 45, 9999]:  # 9999 -> effectively full mean
        m = evaluate_config(train_line_embs, k=k, zscore=zscore)
        results[(k, zscore)] = m
        print(f'k={k:<5} zscore={zscore!s:<5}  MRR@10={m:.3f}')

best_cfg = max(results, key=results.get)
BEST_K, BEST_ZSCORE = best_cfg
print('\nbest config:', best_cfg, '->', results[best_cfg])

k=5     zscore=False  MRR@10=0.515
k=10    zscore=False  MRR@10=0.515
k=15    zscore=False  MRR@10=0.515
k=20    zscore=False  MRR@10=0.515
k=30    zscore=False  MRR@10=0.515
k=45    zscore=False  MRR@10=0.515
k=9999  zscore=False  MRR@10=0.515
k=5     zscore=True   MRR@10=0.515
k=10    zscore=True   MRR@10=0.515
k=15    zscore=True   MRR@10=0.515
k=20    zscore=True   MRR@10=0.515
k=30    zscore=True   MRR@10=0.515
k=45    zscore=True   MRR@10=0.515
k=9999  zscore=True   MRR@10=0.515

best config: (5, False) -> 0.5150452380952381


In [128]:
rows = []
for i, (_, r) in enumerate(test.iterrows()):
    line_emb = test_query_embs[i]
    top10 = rank_top10(line_emb, img_emb_test, gallery_ids, k=BEST_K)
    rows.append({'subtaskID': 1, 'datapointID': int(r['datapointID']), 'answer': ' '.join(top10)})

print('built', len(rows), 'predictions using k =', BEST_K, ', zscore =', BEST_ZSCORE)

built 100 predictions using k = 5 , zscore = False


In [129]:
sub = pd.DataFrame(rows)[['subtaskID', 'datapointID', 'answer']]
sub = sub.sort_values('datapointID').reset_index(drop=True)
sub.to_csv('/content/submission.csv', index=False)
sub.head()

,subtaskID,datapointID,answer
0,1,1,img_00242 img_00268 img_00345 img_00208 img_00...
1,1,2,img_00258 img_00541 img_00420 img_00335 img_00...
2,1,3,img_00563 img_00209 img_00258 img_00557 img_00...
3,1,4,img_00060 img_00400 img_00258 img_00590 img_00...
4,1,5,img_00169 img_00187 img_00180 img_00340 img_00...


#||TRAIN||

In [206]:
for name, param in model.named_parameters():
  if ("11" in name) or ("post_layernorm" in name) or ("final_layer_norm" in name) or ("projection" in name):
    param.requires_grad = True
  else:
    param.requires_grad = False
trainable_params = [param for name, param in model.named_parameters() if param.requires_grad]

optimizer = torch.optim.AdamW(trainable_params, lr=5e-5, weight_decay=1e-5)
model.to('cuda')

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [222]:

rows=[]
for i, r in train.iterrows():
  with torch.no_grad():
    #optimizer.zero_grad()
    q = encode_query([r['description']]).to('cuda')
    sims = (q @ img_emb_train.T).mean(dim=0).cpu() #.to('cuda')
    top10 = [train_ids[j] for j in np.argsort(-sims)] # [:10]
    #print(int(r['image_id'][4:]))
    k = top10.index(r['image_id'])
    print(k)

    loss = 1 - sims[int(r['image_id'][4:])-1]
    #loss.backward()
    #optimizer.step()

    #print('d')
    #print(sims.tolist().index(sims.max()))
    #print(int(r['image_id'][4:])-1)
    #print('m')
    #print(loss)
    #loss.backward()
    #optimizer.step()
    #print(i)
    #rows.append({'subtaskID':1, 'datapointID':r['datapointID'], 'answer':' '.join(top10)})
print('built', len(rows), 'predictions')

275
338
51
107
114
33
69
81
46
87
179
444
10
454
181


In [181]:
int(train['image_id'][1][4:])-1

5